# Lab 04 - Agente Analista Nativo (Gemini + Pandas)
**Disciplina:** Augmented Analytics & AI-Driven Insights

Este notebook demonstra como criar um agente analista que gera e executa código Python em tempo real para explorar dados.

In [ ]:
# 1. Instalação da biblioteca oficial
!pip install -q -U google-generativeai

In [ ]:
import pandas as pd
import google.generativeai as genai
import matplotlib.pyplot as plt
import seaborn as sns

# 2. Configuração da API Key
# INSTRUÇÃO: Substitua 'COLE_SUA_API_KEY_AQUI' pela sua chave do Google AI Studio
API_KEY = 'COLE_SUA_API_KEY_AQUI'
genai.configure(api_key=API_KEY)

In [ ]:
# 3. Carregamento dos dados
try:
    df = pd.read_csv('Lab 04 - adult.csv')
    print("✅ Dataset carregado no estado bruto (sujo)!")

    contexto_colunas = df.dtypes.to_string()
    amostra_dados = df.head(3).to_markdown()
except FileNotFoundError:
    print("❌ Erro: Arquivo não encontrado.")

In [ ]:
import re

def sanitizar_dados():
    """
    Usa a IA para identificar e corrigir problemas de formatação no dataframe global.
    """
    model = genai.GenerativeModel('gemini-3-flash-preview')
    
    prompt = f"""
    Você é um Engenheiro de Dados. O dataframe 'df' possui as seguintes colunas: {contexto_colunas}
    
    Sabe-se que os dados categóricos (texto) possuem espaços em branco extras (ex: ' Female' em vez de 'Female'), o que quebra as análises.
    
    SUA TAREFA:
    1. Escreva um código Python para limpar permanentemente esses espaços de todas as colunas de texto do dataframe 'df'.
    2. Retorne APENAS o código puro dentro de blocos ```python.
    """
    
    try:
        response = model.generate_content(prompt)
        match = re.search(r'```python\n(.*?)\n```', response.text, re.DOTALL)
        if match:
            codigo = match.group(1).strip()
            print("🤖 Agente de Dados: Iniciando saneamento...\n")
            exec(codigo, globals())
            print("✅ Saneamento concluído com sucesso!")
        else:
            print("❌ Erro na extração do código de limpeza.")
    except Exception as e:
        print(f"⚠️ Erro no saneamento: {e}")

def chat_com_dados(pergunta):
    """
    Esta função usa o Gemini para gerar código Pandas e executá-lo localmente.
    """
    model = genai.GenerativeModel('gemini-3-flash-preview')

    prompt = f"""
    Você é um Analista de Dados especializado em Python/Pandas.
    O dataframe 'df' já está carregado e SANEADO na memória.
    Colunas: {contexto_colunas}
    Amostra: {amostra_dados}

    IMPORTANTE: A coluna 'ABOVE50K' é numérica (0 ou 1).

    PERGUNTA DO USUÁRIO: {pergunta}

    SUA TAREFA:
    1. Escreva APENAS o código Python para responder a pergunta.
    2. Use 'print()' para o resultado e 'plt.show()' para gráficos.
    3. Retorne APENAS o código puro dentro de blocos ```python.
    """

    try:
        response = model.generate_content(prompt)
        match = re.search(r'```python\n(.*?)\n```', response.text, re.DOTALL)
        if match:
            codigo = match.group(1).strip()
            print(f"🤖 Analista IA executando: {pergunta}\n")
            exec(codigo, globals())
        else:
            print("❌ Erro na extração do código analítico.")
    except Exception as e:
        print(f"⚠️ Erro na execução: {e}")
        if 'codigo' in locals():
            print(f"Código tentado:\n{codigo}")

In [ ]:
# 5. SANEAMENTO DOS DADOS (Obrigatório antes do Chat)
sanitizar_dados()

In [ ]:
# 6. EXEMPLOS DE USO ANALÍTICO:
chat_com_dados("Qual a média de idade das pessoas que ganham acima de 50k?")

In [ ]:
chat_com_dados("Gere um gráfico de dispersão de Idade vs Horas por Semana colorindo por Sexo.")